# Workbook 16 — SMS Customer Activation Analytics

## Customer Activation System™

### Objective

Analyze the Pizza House Grand Reopening SMS campaign by combining:

- Campaign summary screenshots
- SimpleTexting delivery reports
- Reply screenshots
- Cleaned campaign import files

This workbook turns SMS campaign activity into a measurable customer activation system:

**Customer list → SMS delivery → replies → verified audience → future marketing database**

## 16.0 Business Context

- Pizza House moved to **5050 Stockton Blvd**
- Workbook 13 created a cleaned SMS-ready customer dataset
- SimpleTexting was used to launch a Grand Reopening SMS campaign
- Clover POS does not reliably connect orders to customers
- SMS engagement becomes the first reliable customer validation layer

## 16.1 Setup — File Paths

Define all folders once so the notebook remains portable inside the project repository.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CLEANED_DIR = DATA_DIR / "cleaned"
EXPORT_DIR = DATA_DIR / "exports"

SMS_DIR = RAW_DIR / "sms"
SC_DIR = SMS_DIR / "campaign_summaries"
DR_DIR = SMS_DIR / "delivery_reports"
RES_DIR = SMS_DIR / "replies"
IMPORTS_DIR = SMS_DIR / "imports"

for folder in [RAW_DIR, CLEANED_DIR, EXPORT_DIR, SMS_DIR, SC_DIR, DR_DIR, RES_DIR, IMPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("WB16 folders ready")
print("Campaign summaries:", SC_DIR)
print("Delivery reports:", DR_DIR)
print("Replies:", RES_DIR)
print("Imports:", IMPORTS_DIR)

## 16.2 File Inventory

Confirm that the expected SMS campaign files are available before building the pipeline.

In [ ]:
sc_files = sorted(SC_DIR.glob("*"))
dr_files = sorted(DR_DIR.glob("*.csv"))
res_files = sorted(RES_DIR.glob("*"))

print("Campaign summary files:", len(sc_files))
for file in sc_files:
    print(" -", file.name)

print("\nDelivery report files:", len(dr_files))
for file in dr_files:
    print(" -", file.name)

print("\nReply screenshot files:", len(res_files))
for file in res_files:
    print(" -", file.name)

## 16.3 Campaign Summary Table

SimpleTexting campaign summary screenshots provide campaign-level metrics.

Because the screenshots are not structured CSV exports, this section creates a manual summary table that can be updated from the screenshots.

In [ ]:
campaign_summary = pd.DataFrame([
    {"campaign": "D1_A", "campaign_group": "Initial Test", "contacts": 498, "send_date": "2026-06-08", "send_time": "15:30", "total_sent": np.nan, "success_rate": np.nan, "opt_out_rate": np.nan, "responses": np.nan, "credits_used": np.nan, "notes": "Initial test wave"},
    {"campaign": "D1_B", "campaign_group": "Initial Test", "contacts": 493, "send_date": "2026-06-08", "send_time": "15:45", "total_sent": np.nan, "success_rate": np.nan, "opt_out_rate": np.nan, "responses": np.nan, "credits_used": np.nan, "notes": "Initial test wave"},
    {"campaign": "D2", "campaign_group": "Scale Wave", "contacts": 2468, "send_date": "2026-06-09", "send_time": "various", "total_sent": np.nan, "success_rate": np.nan, "opt_out_rate": np.nan, "responses": np.nan, "credits_used": np.nan, "notes": "Original D2 campaign; partial send resumed later"},
    {"campaign": "D3", "campaign_group": "Restart Wave", "contacts": 996, "send_date": "2026-06-12", "send_time": "15:30", "total_sent": np.nan, "success_rate": np.nan, "opt_out_rate": np.nan, "responses": np.nan, "credits_used": np.nan, "notes": "Restarted clean wave"},
    {"campaign": "D3_2", "campaign_group": "System Constraint Wave", "contacts": np.nan, "send_date": "2026-06-10", "send_time": "various", "total_sent": np.nan, "success_rate": np.nan, "opt_out_rate": np.nan, "responses": np.nan, "credits_used": np.nan, "notes": "Second D3 campaign created during SimpleTexting limit issue"},
    {"campaign": "D4", "campaign_group": "Weekend Wave", "contacts": 1501, "send_date": "2026-06-14", "send_time": "14:00", "total_sent": np.nan, "success_rate": np.nan, "opt_out_rate": np.nan, "responses": np.nan, "credits_used": np.nan, "notes": "Saturday send"},
    {"campaign": "D5", "campaign_group": "Final Wave", "contacts": 1509, "send_date": "2026-06-16", "send_time": "16:00", "total_sent": np.nan, "success_rate": np.nan, "opt_out_rate": np.nan, "responses": np.nan, "credits_used": np.nan, "notes": "Final stretch"},
    {"campaign": "D6", "campaign_group": "Final Wave", "contacts": 1511, "send_date": "2026-06-17", "send_time": "16:00", "total_sent": np.nan, "success_rate": np.nan, "opt_out_rate": np.nan, "responses": np.nan, "credits_used": np.nan, "notes": "Final stretch"}
])

campaign_summary["send_date"] = pd.to_datetime(campaign_summary["send_date"])
campaign_summary

In [ ]:
campaign_summary.to_csv(CLEANED_DIR / "campaign_summary.csv", index=False)
print("Exported:", CLEANED_DIR / "campaign_summary.csv")

## 16.4 Delivery Report Import

Import every delivery report in:

`data/raw/sms/delivery_reports/`

and append them into one master delivery table.

In [ ]:
def campaign_from_delivery_filename(path):
    return path.stem.replace("_dr", "").upper()

delivery_frames = []

for path in sorted(DR_DIR.glob("*_dr.csv")):
    campaign = campaign_from_delivery_filename(path)
    temp = pd.read_csv(path)
    temp.columns = (
        temp.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    temp["campaign"] = campaign
    temp["source_file"] = path.name
    delivery_frames.append(temp)
    print(f"Loaded {campaign}: {len(temp):,} rows")

delivery_master = pd.concat(delivery_frames, ignore_index=True)
delivery_master.head()

## 16.5 Delivery Data Cleaning

Standardize phone numbers and delivery statuses.

In [ ]:
phone_candidates = [col for col in delivery_master.columns if "phone" in col]
phone_col = phone_candidates[0] if phone_candidates else None

if phone_col:
    delivery_master["phone_clean"] = (
        delivery_master[phone_col]
        .astype(str)
        .str.replace(r"\D", "", regex=True)
    )
else:
    delivery_master["phone_clean"] = np.nan

status_candidates = [col for col in delivery_master.columns if "status" in col]
status_col = status_candidates[0] if status_candidates else None

if status_col:
    delivery_master["delivery_status_clean"] = (
        delivery_master[status_col]
        .astype(str)
        .str.strip()
        .str.lower()
    )
else:
    delivery_master["delivery_status_clean"] = np.nan

delivery_master[["campaign", "phone_clean", "delivery_status_clean"]].head()

In [ ]:
delivery_status_summary = (
    delivery_master
    .groupby(["campaign", "delivery_status_clean"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["campaign", "count"], ascending=[True, False])
)

delivery_status_summary

In [ ]:
delivery_master.to_csv(CLEANED_DIR / "delivery_report_master.csv", index=False)
delivery_status_summary.to_csv(CLEANED_DIR / "delivery_status_summary.csv", index=False)

print("Exported:", CLEANED_DIR / "delivery_report_master.csv")
print("Exported:", CLEANED_DIR / "delivery_status_summary.csv")

## 16.6 Reply Screenshot Inventory

SimpleTexting does not export reply text.

Reply screenshots are stored in:

`data/raw/sms/replies/`

In [ ]:
reply_inventory = pd.DataFrame({
    "source_file": [file.name for file in sorted(RES_DIR.glob("*"))],
    "path": [str(file) for file in sorted(RES_DIR.glob("*"))]
})

reply_inventory.to_csv(CLEANED_DIR / "reply_screenshot_inventory.csv", index=False)

reply_inventory

## 16.7 Reply Tracker Template

Replies will be transcribed from screenshots into a structured tracker.

In [ ]:
reply_tracker_columns = [
    "campaign",
    "source_file",
    "contact_display",
    "reply_text",
    "reply_category",
    "coupon_sent",
    "follow_up_needed",
    "notes"
]

reply_tracker = pd.DataFrame(columns=reply_tracker_columns)
reply_tracker.to_csv(CLEANED_DIR / "reply_tracker_template.csv", index=False)

print("Exported:", CLEANED_DIR / "reply_tracker_template.csv")

## 16.8 Reply Category Rules

Suggested categories:

- YES
- QUESTION
- HELP
- NEGATIVE
- WRONG_NUMBER
- OTHER
- STOP

In [ ]:
def classify_reply(text):
    if pd.isna(text):
        return "UNKNOWN"

    value = str(text).strip().lower()
    yes_values = {"yes", "y", "yeah", "yep", "si", "sí", "ok", "okay"}

    if value in yes_values or value.startswith("yes"):
        return "YES"
    if "stop" in value:
        return "STOP"
    if "help" in value:
        return "HELP"
    if "wrong" in value or "remove" in value:
        return "WRONG_NUMBER"
    if any(term in value for term in ["address", "cross", "where", "hours", "open", "location"]):
        return "QUESTION"
    if value in {"no", "nah", "nope"} or "don't care" in value:
        return "NEGATIVE"

    return "OTHER"

example_replies = ["YES", "What's the cross street?", "STOP", "wrong number", "HELP", "No", "LUGIS"]
pd.DataFrame({
    "reply_text": example_replies,
    "reply_category": [classify_reply(x) for x in example_replies]
})

## 16.9 Data Quality Outputs

Create reusable customer cleanup outputs from delivery data:

- invalid_numbers.csv
- duplicate_phones.csv

In [ ]:
invalid_keywords = ["invalid", "failed", "undelivered", "error"]

invalid_numbers = delivery_master[
    delivery_master["delivery_status_clean"].astype(str).str.contains("|".join(invalid_keywords), na=False)
].copy()

duplicate_phones = (
    delivery_master[delivery_master["phone_clean"].notna()]
    .groupby("phone_clean")
    .size()
    .reset_index(name="record_count")
    .query("record_count > 1")
    .sort_values("record_count", ascending=False)
)

invalid_numbers.to_csv(CLEANED_DIR / "invalid_numbers.csv", index=False)
duplicate_phones.to_csv(CLEANED_DIR / "duplicate_phones.csv", index=False)

print("Invalid numbers:", len(invalid_numbers))
print("Duplicate phones:", len(duplicate_phones))

## 16.10 Campaign Funnel

This section creates the first version of the SMS activation funnel from available delivery data.

Reply and YES metrics will be added after `reply_tracker.csv` is completed.

In [ ]:
total_delivery_records = len(delivery_master)
unique_phone_records = delivery_master["phone_clean"].nunique()

funnel = pd.DataFrame([
    {"stage": "Delivery Report Records", "count": total_delivery_records},
    {"stage": "Unique Phone Numbers", "count": unique_phone_records},
    {"stage": "Invalid / Failed Records", "count": len(invalid_numbers)},
])

funnel.to_csv(EXPORT_DIR / "sms_activation_funnel.csv", index=False)

funnel

## 16.11 Tableau Export Plan

Final export files:

- campaign_summary.csv
- delivery_report_master.csv
- reply_tracker.csv
- customer_master.csv
- wb16_sms_customer_activation.csv

## 16.12 Next Steps

1. Update campaign summary metrics from SimpleTexting screenshots.
2. Transcribe reply screenshots into `reply_tracker.csv`.
3. Merge delivery + reply data into `customer_master.csv`.
4. Build yes / opt-out / question / invalid customer lists.
5. Export Tableau-ready dataset.
6. Add executive findings and README update.